In [ ]:
from qwen_tts import Qwen3TTSModel
import sounddevice as sd
import soundfile as sf
import numpy as np

In [1]:
from pathlib import Path

In [8]:
datapath = Path.cwd().parent / "data"
reference_raw = str(datapath / "reference_raw.wav")
reference = str(datapath / "reference.wav")

In [ ]:
tts = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base")

In [ ]:
import sounddevice as sd

print(sd.query_devices())
print("\nDefault input device:", sd.default.device)


In [ ]:
import sounddevice as sd
import soundfile as sf
import librosa

MIC_DEVICE = 2  # MacBook Pro Microphone


def record_wav(filename=reference_raw, duration=8):
    device_info = sd.query_devices(MIC_DEVICE)
    samplerate = int(device_info["default_samplerate"])
    print(f"🎙️  Recording {duration}s at {samplerate}Hz on '{device_info['name']}'...")
    audio = sd.rec(
        int(duration * samplerate),
        samplerate=samplerate,
        channels=1,
        dtype="float32",
        device=MIC_DEVICE,
    )
    sd.wait()
    sf.write(filename, audio, samplerate, subtype="PCM_16")
    print(f"✅ Saved raw recording to {filename}")
    return filename


def resample_to_24k(input_file, output_file=reference):
    audio, sr = librosa.load(input_file, sr=None, mono=True)
    audio_24k = librosa.resample(audio, orig_sr=sr, target_sr=24000)
    sf.write(output_file, audio_24k, 24000, subtype="PCM_16")
    print(f"✅ Resampled {sr}Hz → 24000Hz: {output_file}")
    return output_file


# Full pipeline
raw = record_wav(reference_raw, duration=8)
final = resample_to_24k(raw, reference)


In [ ]:
prompt = tts.create_voice_clone_prompt(
    ref_audio=reference,
    ref_text="testing testing 1, 2, 3. testing, testing, 3,4,5.",
    x_vector_only_mode=False,  # use full ICL for best quality
)

wavs, sr = tts.generate_voice_clone(
    text="hello my name is peter",
    language="English",
    voice_clone_prompt=prompt,
)


In [ ]:
from IPython.display import Audio, display

display(Audio(wavs[0], rate=sr))
